# Regulatory Compliance Multi-Agent System
## Capstone Assignment - 8-Week Progressive Exercise

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Assignment Overview

In this capstone assignment, you will build a **production-grade multi-agent regulatory compliance system** for the insurance industry that analyzes company policies against regulatory requirements, identifies compliance gaps, and provides remediation recommendations.

### What You Will Build

| Component | Description |
|-----------|-------------|
| **PDF/DOCX Regulation Parser** | Extract text and tables from regulatory documents |
| **Requirement Extractor** | Use regex patterns to identify mandatory requirements |
| **Gap Detector Agent** | Compare regulations against company policies |
| **Compliance Scoring System** | Calculate domain-level compliance scores |
| **Regulatory Knowledge Graph** | Map relationships between regulations, domains, and policies |

### Dataset Overview

| Category | Files | Description |
|----------|-------|-------------|
| **Regulatory PDFs** | 14 PDFs | IRDAI regulations, MAS guidelines, audit controls |
| **Company Policies** | 8 DOCX | Internal policies (Governance, Risk, Underwriting, Cyber) |
| **Regulatory Sources** | 1 DOCX | Reference list of regulatory sources |

### 8-Week Structure

| Week | Module | Difficulty |
|------|--------|------------|
| 1 | Environment & Data Loading | Beginner |
| 2 | Document Processing & EDA | Beginner |
| 3 | Vector Store & Embeddings | Intermediate |
| 4 | Single Agent Design | Intermediate |
| 5 | Multi-Agent Orchestration | Advanced |
| 6 | Compliance Knowledge Graph | Advanced |
| 7 | Advanced Observability | Expert |
| 8 | Production Deployment | Expert |

### Grading Rubric

- **Functionality (40%)**: All components work correctly
- **Code Quality (20%)**: Clean, documented, follows best practices
- **Observability (20%)**: Proper Langfuse tracing throughout
- **Compliance Domain (20%)**: Correct regulatory logic and gap detection

---
# WEEK 1: Environment Setup & Data Loading
---

## Learning Objectives
- Configure development environment in Google Colab
- Initialize Langfuse observability from the start
- Load and explore regulatory documents

## Difficulty: Beginner

**HINT**: Environment setup is foundational. Pay attention to:
- Colab vs local environment detection
- Secure API key handling
- Data directory path configuration for regulations and policies

In [ ]:
# ============================================================================
# WEEK 1.1: PACKAGE INSTALLATION
# ============================================================================
# Pinned versions for reproducibility. Every package is verified after install.

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "pdfplumber", "PyPDF2", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("pdfplumber", "pdfplumber"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

In [ ]:
# ============================================================================
# IMPORTS AND ENVIRONMENT DETECTION (PROVIDED)
# ============================================================================

import os
import sys
import json
import re
import subprocess
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Environment Detection
IN_COLAB = 'google.colab' in sys.modules

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "regulatory_compliance"

PROJECT_NAME = "regulatory-compliance-system"
PROJECT_VERSION = "1.0.0"

if IN_COLAB:
    print("Configuring for Google Colab environment...")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Configuring for local environment...")
    from dotenv import load_dotenv
    load_dotenv()

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)

REGULATIONS_DIR_BASE = f'{DATA_DIR_BASE}/Regulations'
POLICIES_DIR_BASE = f'{DATA_DIR_BASE}/Company Policies sample documents'

# Convert to Path objects for easier manipulation
REGULATIONS_DIR = Path(REGULATIONS_DIR_BASE)
POLICIES_DIR = Path(POLICIES_DIR_BASE)

# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

In [ ]:
# ============================================================================
# TODO 1.1: INITIALIZE LANGFUSE CLIENT
# ============================================================================
#
# Your task: Initialize Langfuse for observability tracking
#
# INSTRUCTIONS:
# 1. Import Langfuse and LangfuseCallbackHandler
# 2. Create a Langfuse client with proper credentials
# 3. Verify the connection with auth_check()
# 4. Create a unique SESSION_ID for this notebook run
#
# HINT: Use os.environ.get() to retrieve credentials
#
# EXPECTED OUTPUT:
# - "Langfuse connected successfully!"
# - Session ID in format: compliance-YYYYMMDD-HHMMSS
# ============================================================================

from langfuse import Langfuse
from langfuse.callback import CallbackHandler as LangfuseCallbackHandler
from openai import OpenAI

# TODO: Initialize Langfuse client
# langfuse = Langfuse(
#     secret_key=...,
#     public_key=...,
#     host=...
# )

# YOUR CODE HERE
langfuse = None  # Replace with your implementation

# TODO: Verify connection
# langfuse.auth_check()

# TODO: Create session ID
SESSION_ID = f"compliance-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Initialize OpenAI client
openai_client = OpenAI()

print(f"Session ID: {SESSION_ID}")

In [ ]:
# ============================================================================
# VALIDATION CELL 1.1: Verify Langfuse Setup
# ============================================================================

def validate_langfuse_setup():
    """Validate that Langfuse is properly configured."""
    errors = []

    if langfuse is None:
        errors.append("Langfuse client not initialized")
    else:
        try:
            langfuse.auth_check()
        except Exception as e:
            errors.append(f"Langfuse auth failed: {e}")

    if not SESSION_ID.startswith("compliance-"):
        errors.append("SESSION_ID should start with 'compliance-'")

    if errors:
        print("VALIDATION FAILED:")
        for e in errors:
            print(f"  - {e}")
        return False

    print("VALIDATION PASSED: Langfuse setup correct!")
    return True

validate_langfuse_setup()

In [ ]:
# ============================================================================
# TODO 1.2: CREATE TRACED HELPER FUNCTIONS
# ============================================================================
#
# Your task: Implement wrapper functions that trace all LLM operations
#
# INSTRUCTIONS:
# 1. Implement traced_embedding() - generates embeddings with Langfuse tracing
# 2. Implement traced_completion() - generates completions with Langfuse tracing
# 3. Implement get_langfuse_handler() - returns LangChain callback handler
#
# COMPLIANCE HINT: Every LLM call in regulatory compliance MUST be traced for:
# - Audit compliance requirements
# - Reproducibility of gap assessments
# - Cost tracking per regulation analysis
#
# EXPECTED OUTPUT:
# Each function should log to Langfuse with proper metadata
# ============================================================================

def traced_embedding(text: str, trace_name: str = "embedding", metadata: Dict = None) -> List[float]:
    """
    Generate embedding with full Langfuse tracing.

    Args:
        text: The text to embed
        trace_name: Name for the Langfuse trace
        metadata: Optional metadata dictionary

    Returns:
        List of floats representing the embedding vector

    TODO:
    1. Create a Langfuse trace with session_id and metadata
    2. Create a generation span for the embedding call
    3. Call OpenAI embeddings API (model: text-embedding-3-small)
    4. End the generation with usage stats
    5. Return the embedding
    """
    # YOUR CODE HERE

    # HINT: Structure your code like this:
    # trace = langfuse.trace(name=trace_name, session_id=SESSION_ID, tags=["embedding", "compliance"], metadata=metadata or {})
    # generation = trace.generation(name="openai-embedding", model="text-embedding-3-small", input=text[:500])
    # response = openai_client.embeddings.create(model="text-embedding-3-small", input=text[:8000])
    # embedding = response.data[0].embedding
    # generation.end(output={"dimensions": len(embedding)}, usage={"input": response.usage.prompt_tokens})
    # return embedding

    pass  # Replace with your implementation


def traced_completion(prompt: str, trace_name: str = "completion",
                      model: str = "gpt-4o-mini", metadata: Dict = None) -> str:
    """
    Generate completion with full Langfuse tracing.

    Args:
        prompt: User prompt
        trace_name: Name for the trace
        model: OpenAI model to use
        metadata: Optional metadata

    Returns:
        The completion text

    TODO:
    1. Create a Langfuse trace
    2. Create generation span
    3. Call OpenAI chat completions API
    4. End generation with output and token usage
    5. Return result
    """
    # YOUR CODE HERE
    pass  # Replace with your implementation


def get_langfuse_handler(trace_name: str, tags: List[str] = None) -> LangfuseCallbackHandler:
    """
    Get a LangChain callback handler for automatic tracing.

    Args:
        trace_name: Name for the trace
        tags: Optional list of tags

    Returns:
        LangfuseCallbackHandler instance

    TODO:
    Return a LangfuseCallbackHandler with proper credentials and session_id
    """
    # YOUR CODE HERE
    pass  # Replace with your implementation


print("Traced helper functions defined (implement the TODOs above)")

In [ ]:
# ============================================================================
# TODO 1.3: DEFINE COMPLIANCE TAXONOMY
# ============================================================================
#
# Your task: Define the compliance domain taxonomy
#
# INSTRUCTIONS:
# Complete the COMPLIANCE_DOMAINS dictionary with:
# 1. Description of each domain
# 2. Related regulations for each domain
# 3. Related policies for each domain
# 4. Risk weight (0.0 to 1.0) indicating importance
#
# COMPLIANCE DOMAINS TO COVER:
# - governance: Board oversight, corporate structure
# - risk_management: Risk frameworks, assessments
# - information_security: Cybersecurity, data protection
# - claims_management: Claims processing, grievances
# - reinsurance: Reinsurance operations
# - audit: Internal audit, controls
# - sales_distribution: Sales practices, conduct
# - investments: Investment management
# - fraud: Fraud prevention
# ============================================================================

COMPLIANCE_DOMAINS = {
    'governance': {
        'description': 'Corporate governance and structure',
        'regulations': ['corporate-governance-guidelines'],
        'policies': ['Governance and Corporate Structure'],
        'risk_weight': 0.9
    },
    'risk_management': {
        'description': 'Risk management frameworks',
        'regulations': [],  # TODO: Add relevant regulations
        'policies': [],  # TODO: Add relevant policies
        'risk_weight': 0.95
    },
    # TODO: Add remaining domains:
    # - information_security
    # - claims_management
    # - reinsurance
    # - audit
    # - sales_distribution
    # - investments
    # - fraud
}

print("Compliance Domains defined:")
for domain, info in COMPLIANCE_DOMAINS.items():
    print(f"  - {domain}: {info['description']} (risk weight: {info['risk_weight']})")

In [ ]:
# ============================================================================
# TODO 1.4: DISCOVER REGULATORY DATA FILES
# ============================================================================
#
# Your task: Implement a function to discover all regulatory documents
#
# INSTRUCTIONS:
# 1. Search REGULATIONS_DIR for PDF files
# 2. Search POLICIES_DIR for DOCX files
# 3. Return dictionary with 'regulations' and 'policies' keys
#
# COMPLIANCE HINT: Document inventory is essential for:
# - Regulatory audit trails
# - Coverage analysis
# - Gap identification
#
# EXPECTED OUTPUT:
# Dictionary with lists of Path objects for regulations and policies
# ============================================================================

def discover_compliance_data(regulations_dir: Path, policies_dir: Path) -> Dict:
    """
    Discover all regulatory and policy documents.

    Args:
        regulations_dir: Directory containing regulatory PDFs
        policies_dir: Directory containing policy DOCX files

    Returns:
        Dictionary with 'regulations' and 'policies' lists
    """
    discovered = {'regulations': [], 'policies': []}

    # TODO: Discover regulation PDFs
    # HINT: Use regulations_dir.glob('*.pdf')
    if regulations_dir.exists():
        # YOUR CODE HERE
        pass

    # TODO: Discover policy DOCX files
    # HINT: Use policies_dir.glob('*.docx')
    if policies_dir.exists():
        # YOUR CODE HERE
        pass

    return discovered


# Discover documents with tracing
if langfuse:
    trace = langfuse.trace(
        name="discover-documents",
        session_id=SESSION_ID,
        tags=["discovery", "documents"]
    )

compliance_files = discover_compliance_data(REGULATIONS_DIR, POLICIES_DIR)

print("\nREGULATORY COMPLIANCE DOCUMENTS")
print("=" * 60)
print(f"\nRegulations (PDFs): {len(compliance_files['regulations'])}")
for f in compliance_files['regulations'][:5]:
    print(f"  - {f.name}")
if len(compliance_files['regulations']) > 5:
    print(f"  ... and {len(compliance_files['regulations']) - 5} more")

print(f"\nCompany Policies (DOCX): {len(compliance_files['policies'])}")
for f in compliance_files['policies']:
    print(f"  - {f.name}")

---
# WEEK 2: Document Processing & EDA
---

## Learning Objectives
- Parse regulatory PDFs and policy DOCX files
- Extract structured information from documents
- Implement requirement extraction with regex

## Difficulty: Beginner to Intermediate

**COMPLIANCE CHALLENGE**: Regulatory documents have:
- Legal language with "shall", "must", "required" keywords
- Complex nested requirements
- Tables with compliance criteria
- Section/subsection hierarchies

In [ ]:
# ============================================================================
# TODO 2.1: IMPLEMENT REGULATORY DOCUMENT PROCESSOR
# ============================================================================
#
# Your task: Build a document processor for regulatory PDFs and policy DOCX
#
# INSTRUCTIONS:
# 1. Implement load_pdf() - parse PDF documents using pdfplumber
# 2. Implement load_docx() - parse DOCX documents using python-docx
# 3. Implement process_all() - process all discovered documents
#
# COMPLIANCE HINT: Regulatory documents often contain:
# - Tables with compliance requirements
# - Section headers indicating requirement categories
# - Legal language patterns (shall, must, required)
#
# EXPECTED OUTPUT:
# Dict with keys: filename, format, doc_type, full_text, tables, page_count/section_count
# ============================================================================

import pdfplumber
from docx import Document as DocxDocument

class RegulatoryDocumentProcessor:
    """
    Process regulatory PDFs and company policy documents.
    All operations traced to Langfuse.
    """

    def __init__(self):
        self.processed_docs = []

    def load_pdf(self, file_path: Path) -> Dict:
        """
        Extract text and tables from regulatory PDF.

        TODO:
        1. Open PDF with pdfplumber
        2. Extract text from each page
        3. Extract tables from each page
        4. Return structured dictionary

        HINT:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                tables = page.extract_tables()
        """
        try:
            # YOUR CODE HERE
            text_content = []
            tables = []

            # TODO: Extract content from PDF

            result = {
                'filename': file_path.name,
                'format': 'pdf',
                'doc_type': 'regulation',
                'full_text': '\n'.join(text_content),
                'tables': tables,
                'page_count': len(text_content),
                'char_count': sum(len(t) for t in text_content),
                'table_count': len(tables)
            }
            return result
        except Exception as e:
            print(f"Error loading PDF {file_path.name}: {e}")
            return None

    def load_docx(self, file_path: Path) -> Dict:
        """
        Extract text and tables from policy DOCX.

        TODO:
        1. Open document with python-docx
        2. Extract all paragraphs (filter empty ones)
        3. Extract all tables as lists of rows
        4. Return structured dictionary

        HINT for tables:
        for table in doc.tables:
            for row in table.rows:
                row_data = [cell.text.strip() for cell in row.cells]
        """
        try:
            # YOUR CODE HERE
            doc = DocxDocument(file_path)
            paragraphs = []  # TODO: Extract non-empty paragraphs
            tables = []  # TODO: Extract table data

            full_text = '\n'.join(paragraphs)
            result = {
                'filename': file_path.name,
                'format': 'docx',
                'doc_type': 'policy',
                'full_text': full_text,
                'tables': tables,
                'section_count': len(paragraphs),
                'char_count': len(full_text),
                'table_count': len(tables)
            }
            return result
        except Exception as e:
            print(f"Error loading DOCX {file_path.name}: {e}")
            return None

    def process_all(self, files: Dict) -> Dict:
        """
        Process all regulatory and policy documents.

        TODO:
        1. Process all regulation PDFs
        2. Process all policy DOCX files
        3. Assign doc_id to each document (REG-001, POL-001, etc.)
        4. Add Langfuse tracing
        """
        processed = {'regulations': [], 'policies': []}

        # TODO: Process regulations
        for pdf_path in files.get('regulations', []):
            doc = self.load_pdf(pdf_path)
            if doc:
                doc['doc_id'] = f"REG-{len(processed['regulations'])+1:03d}"
                processed['regulations'].append(doc)

        # TODO: Process policies
        for docx_path in files.get('policies', []):
            doc = self.load_docx(docx_path)
            if doc:
                doc['doc_id'] = f"POL-{len(processed['policies'])+1:03d}"
                processed['policies'].append(doc)

        return processed


# Initialize and process documents
doc_processor = RegulatoryDocumentProcessor()
all_compliance_docs = doc_processor.process_all(compliance_files)

print(f"Processed {len(all_compliance_docs['regulations'])} regulations")
print(f"Processed {len(all_compliance_docs['policies'])} policies")

In [ ]:
# ============================================================================
# TODO 2.2: EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================================
#
# Your task: Create EDA visualizations for compliance documents
#
# INSTRUCTIONS:
# 1. Create DataFrames from processed documents
# 2. Generate summary statistics
# 3. Create visualizations showing document distribution
#
# EXPECTED VISUALIZATIONS:
# - Document count by type (bar chart)
# - Document size distribution (bar chart)
# - Domain coverage (grouped bar chart)
# ============================================================================

import matplotlib.pyplot as plt

# Create EDA dataframes
reg_data = []
for doc in all_compliance_docs['regulations']:
    reg_data.append({
        'doc_id': doc['doc_id'],
        'filename': doc['filename'][:40],
        'pages': doc.get('page_count', 0),
        'chars': doc.get('char_count', 0),
        'tables': doc.get('table_count', 0),
        'type': 'Regulation'
    })

pol_data = []
for doc in all_compliance_docs['policies']:
    pol_data.append({
        'doc_id': doc['doc_id'],
        'filename': doc['filename'][:40],
        'sections': doc.get('section_count', 0),
        'chars': doc.get('char_count', 0),
        'tables': doc.get('table_count', 0),
        'type': 'Policy'
    })

df_regs = pd.DataFrame(reg_data)
df_pols = pd.DataFrame(pol_data)

# TODO: Create visualizations
# HINT: Use matplotlib to create 2x2 subplot figure
# Plot 1: Document counts by type
# Plot 2: Regulation sizes
# Plot 3: Policy sizes
# Plot 4: Domain coverage

print("REGULATORY DOCUMENTS ANALYSIS")
print("=" * 60)
if not df_regs.empty:
    print(df_regs[['doc_id', 'filename', 'pages', 'chars', 'tables']].to_string(index=False))
print(f"\nTotal regulations: {len(df_regs)}")

print("\n" + "=" * 60)
print("POLICY DOCUMENTS ANALYSIS")
print("=" * 60)
if not df_pols.empty:
    print(df_pols[['doc_id', 'filename', 'sections', 'chars', 'tables']].to_string(index=False))
print(f"\nTotal policies: {len(df_pols)}")

In [ ]:
# ============================================================================
# TODO 2.3: IMPLEMENT REQUIREMENT EXTRACTOR
# ============================================================================
#
# Your task: Extract regulatory requirements using regex patterns
#
# INSTRUCTIONS:
# 1. Define regex patterns for requirement keywords (shall, must, required)
# 2. Implement extract_requirements() to find requirement statements
# 3. Implement categorize_requirement() to map requirements to domains
#
# COMPLIANCE HINT - Common requirement patterns:
# - "shall" indicates mandatory obligation
# - "must" indicates strict requirement
# - "required to" indicates compliance obligation
#
# EXAMPLE PATTERNS:
# r'(?:shall|must|required to)\s+(.+?)(?:\.|;|$)'
# ============================================================================

class RequirementExtractor:
    """
    Extract regulatory requirements from documents.
    """

    def __init__(self):
        # TODO: Define regex patterns for requirements
        self.requirement_patterns = [
            r'(?:shall|must|required to)\s+(.+?)(?:\.|;|$)',
            # TODO: Add more patterns:
            # - Pattern for "the insurer/company shall..."
            # - Pattern for "mandatory requirement..."
            # - Pattern for "obligated to..."
        ]

    def extract_requirements(self, text: str, doc_id: str = None) -> List[str]:
        """
        Extract requirement statements from text.

        TODO:
        1. Iterate through requirement_patterns
        2. Use re.findall() to find all matches
        3. Filter requirements by length (20-500 chars)
        4. Return unique requirements
        """
        requirements = []
        # YOUR CODE HERE

        # HINT:
        # for pattern in self.requirement_patterns:
        #     matches = re.findall(pattern, text, re.IGNORECASE)
        #     for match in matches:
        #         req = match.strip()
        #         if 20 < len(req) < 500:
        #             requirements.append(req)

        return list(set(requirements))[:50]

    def categorize_requirement(self, requirement: str) -> str:
        """
        Categorize a requirement by compliance domain.

        TODO:
        Match requirement text against domain keywords:
        - governance: board, director, governance, corporate, committee
        - risk_management: risk, assessment, mitigation, exposure
        - information_security: security, cyber, data, privacy, breach
        - audit: audit, control, internal, review, assurance
        - fraud: fraud, misrepresentation, investigation
        - reinsurance: reinsurance, cede, retrocession, treaty
        - claims: claim, settlement, grievance, redressal
        """
        req_lower = requirement.lower()

        # TODO: Implement domain matching
        # YOUR CODE HERE

        return 'general'

    def process_document(self, doc: Dict) -> Dict:
        """
        Extract and categorize requirements from a document.
        """
        text = doc.get('full_text', '')
        requirements = self.extract_requirements(text, doc.get('doc_id'))

        categorized = {}
        for req in requirements:
            category = self.categorize_requirement(req)
            if category not in categorized:
                categorized[category] = []
            categorized[category].append(req)

        return {
            'doc_id': doc.get('doc_id'),
            'filename': doc.get('filename'),
            'total_requirements': len(requirements),
            'requirements_by_category': categorized
        }


# Initialize and test
req_extractor = RequirementExtractor()
print("RequirementExtractor initialized")

In [ ]:
# ============================================================================
# VALIDATION CELL 2.1: Test Requirement Extraction
# ============================================================================

def validate_requirement_extraction():
    """Test the requirement extractor with sample regulatory text."""

    test_text = """
    The Board of Directors shall establish a risk management committee.
    The insurer must maintain adequate capital reserves.
    Companies are required to submit quarterly compliance reports.
    All data must be encrypted in transit and at rest.
    """

    requirements = req_extractor.extract_requirements(test_text)

    print("Requirement Extraction Test Results:")
    print(f"  Requirements found: {len(requirements)}")
    for req in requirements:
        category = req_extractor.categorize_requirement(req)
        print(f"    - [{category}] {req[:60]}...")

    # Validation checks
    passed = True
    if len(requirements) < 2:
        print("  WARNING: Should find at least 2 requirements")
        passed = False

    if passed:
        print("\nVALIDATION PASSED!")
    else:
        print("\nVALIDATION NEEDS WORK - implement the TODO sections above")

validate_requirement_extraction()